In [10]:
# =========================
# 🔧 INSTALL (STABLE SETUP)
# =========================
!pip install -q langchain langchain-community langchain-core
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu pypdf sentence-transformers
!pip install -q groq gradio requests

# =========================
# 📥 DOWNLOAD PDF
# =========================
import requests

file_id = "1Q4EUHwH9yotWAR3S9QaktvAI-tV6qFW3"
url = f"https://drive.google.com/uc?id={file_id}"

pdf_path = "document.pdf"

response = requests.get(url)
with open(pdf_path, "wb") as f:
    f.write(response.content)

print("✅ PDF downloaded")

# =========================
# 📄 LOAD & SPLIT PDF
# =========================
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Total chunks: {len(chunks)}")

# =========================
# 🧠 EMBEDDINGS
# =========================
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# =========================
# 🗄️ FAISS VECTOR DB
# =========================
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

# =========================
# ⚡ GROQ SETUP
# =========================
from groq import Groq

# 🔑 PUT YOUR GROQ API KEY HERE
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = Groq()

# =========================
# 🔍 RETRIEVAL
# =========================
def retrieve_docs(query, k=3):
    docs = vectorstore.similarity_search(query, k=k)
    return docs

# =========================
# 🤖 GENERATION WITH FALLBACK
# =========================
def generate_answer(query):
    try:
        docs = retrieve_docs(query)

        if not docs:
            return "❌ No relevant content found."

        context = "\n\n".join([doc.page_content for doc in docs])

        prompt = f"""
Answer ONLY from the context below.
If not found, say "I don't know".

Context:
{context}

Question:
{query}
"""

        # ✅ Try multiple models (auto fallback)
        models = [
            "mixtral-8x7b-32768",
            "gemma2-9b-it",
            "llama-3.3-70b-versatile"
        ]

        for model_name in models:
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.3
                )
                return f"✅ (Model: {model_name})\n\n" + response.choices[0].message.content

            except Exception as model_error:
                continue

        return "❌ All models failed. Check your Groq API access."

    except Exception as e:
        return f"❌ Error: {str(e)}"

# =========================
# 🌐 GRADIO UI
# =========================
import gradio as gr

def rag_chat(query):
    if not query.strip():
        return "⚠️ Please enter a question."
    return generate_answer(query)

iface = gr.Interface(
    fn=rag_chat,
    inputs=gr.Textbox(lines=2, placeholder="Ask something from the PDF..."),
    outputs="text",
    title="📚 RAG App (Stable Version)",
    description="Uses fallback models to avoid errors"
)

iface.launch(share=True)

✅ PDF downloaded
✅ Total chunks: 25


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fe6743b22702c159a4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
